> 在线讲义：[第 43 章 · 从通信改进到任务完成：先看谁晚到，再看能藏多少](https://wikieden.github.io/ai-infra-from-constraints/lessons/43-task-completion.html) · 本书《从约束推导 AI Infra》

# 第 43 章 · 从通信改进到任务完成：先看谁晚到，再看能藏多少

配套讲义：仓库里 `ai-infra/notes/lessons/43-task-completion.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



就绪偏差、例题 7.6 的六种安排、训练与推理的启动 / 带宽、1024 卡超节点表、故障开销取自 AII 第 7 章 §7.6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MiB = 2**20
A = 0.83e-6
NIC, NVL = 50e9, 450e9

**函数 `ring_continuous_ms`**：连续环（同第 39 章）：每轮等跨服务器那条边。

In [ ]:
def ring_continuous_ms(M: float = 192 * MiB, p: int = 16, bw: float = NIC) -> float:
    """连续环（同第 39 章）：每轮等跨服务器那条边。"""
    return 2 * (p - 1) * (A + M / p / bw) * 1e3

**函数 `hierarchical_ms`**：分层归约（同第 39 章）。

In [ ]:
def hierarchical_ms(M: float = 192 * MiB, local: int = 8, servers: int = 2, BL: float = NVL, BN: float = NIC) -> float:
    """分层归约（同第 39 章）。"""
    shard = M / local
    t_local = 2 * (local - 1) * shard / BL
    t_remote = 2 * (servers - 1) * shard / servers / BN
    return (t_local + t_remote + (2 * (local - 1) + 2 * (servers - 1)) * A) * 1e3

**函数 `step_ms`**：ready=None 时计算、通信、更新串行；否则通信从 ready 开始与剩余计算重叠。

In [ ]:
def step_ms(comm: float, ready: float | None = None, comp: float = 20, upd: float = 2) -> float:
    """ready=None 时计算、通信、更新串行；否则通信从 ready 开始与剩余计算重叠。"""
    if ready is None:
        return comp + comm + upd
    return max(comp, ready + comm) + upd

**函数 `ar8_us`**：小工具。

In [ ]:
def ar8_us(M: float, alpha: float = 5e-6, B: float = 50e9) -> float:
    return (14 * alpha + 1.75 * M / B) * 1e6

**函数 `t_grad`**：小工具。

In [ ]:
def t_grad(S: int, exit_bw: float | None = None, G: float = 8e9, BL: float = 450e9, BN: float = 50e9) -> float:
    q, H = S // 8, 1024 // S
    v_out = 2 * (H - 1) / H * 8 * G
    b_out = exit_bw if exit_bw else S * BN
    local = 2 * (q - 1) * A + 2 * (q - 1) * G / (q * BL)
    remote = 2 * (H - 1) * A + max(2 * (H - 1) * G / (H * q * BN), v_out / b_out)
    hier = local + remote
    ring = 2 * 127 * (A + G / 128 / BN)  # 不分层的连续 DP 环
    return min(hier, ring)

**函数 `step_1024`**：小工具。

In [ ]:
def step_1024(S: int, exit_bw: float | None = None) -> tuple[float, float]:
    t = 0.485 + 0.086 + t_grad(S, exit_bw) + 0.050
    return t, 2**20 / t

**函数 `fault_overhead`**：小工具。

In [ ]:
def fault_overhead(S: int, C: float = 10, I: float = 600, mtbf_h: float = 7.9) -> float:
    lam = 1 / (mtbf_h * 3600)
    R = 60 + S * 64e9 / 64e9
    return C / I + lam * (I / 2 + R)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：就绪偏差：0、0、0、2 ms 就绪，交换 0.4 ms")
print(f"  {2 + 0.4} ms；交换减半 {2 + 0.2} ms；同时就绪 0.4 ms")

In [ ]:
print("\n第二步：例题 7.6（192 MiB 梯度）")
ring, hier = ring_continuous_ms(), hierarchical_ms()
hier_lowslot = hier - 2 * 12 * MiB / 50e9 * 1e3 + 24 * MiB / 16.384e9 * 1e3
one_nic = hier - 2 * 12 * MiB / 50e9 * 1e3 + 2 * 96 * MiB / 50e9 * 1e3
for name, c, r in (("连续环串行", ring, None), ("分层串行", hier, None), ("分层 128 槽位", hier_lowslot, None),
                   ("连续环 17 ms 就绪", ring, 17), ("分层 17 ms 就绪", hier, 17), ("只剩一张网卡", one_nic, 17)):
    print(f"  {name}：通信 {c:.1f} ms，整步 {step_ms(c, r):.1f} ms")
print(f"  分层通信最晚 {20 - hier:.1f} ms 就绪仍能藏住")

In [ ]:
print("\n第三步：训练与推理")
for M in (8 * MiB, 8 * 1024):
    print(f"  {M / 1024:,.0f} KiB：50 GB/s {ar8_us(M):.1f} μs，150 GB/s {ar8_us(M, B=150e9):.1f} μs")
print(f"  72 次 8 KiB：α 5 → 2 μs 省 {72 * 14 * 3 / 1e3:.2f} ms；带宽 ×3 省 {72 * (ar8_us(8192) - ar8_us(8192, B=150e9)):.0f} μs")
print(f"  启动 = 传输的数据量 {14 * 5e-6 * 50e9 / 1.75 / 1e6:.0f} MB")
print(f"  V4.1 迁移 128K 全局 KV（25 GB/s）：V4-Flash {439.281 * MiB / 25e9 * 1e3:.3f} ms，V4.1 {111.25 * MiB / 25e9 * 1e3:.3f} ms")

In [ ]:
print("\n第四步：1024 卡 TP8×DP128")
for S in (8, 64, 128, 256):
    t1, th1 = step_1024(S)
    t2, th2 = step_1024(S, 400e9)
    print(f"  {S:3d} 卡超节点：出口随卡数 {t1:.3f} s / {th1 / 1e4:.1f} 万 token/s；出口 400 GB/s {t2:.3f} s / {th2 / 1e4:.1f} 万")
for S in (128, 256):
    e = fault_overhead(S)
    print(f"  {S} 卡：故障开销 {e:.1%}，有效吞吐 {step_1024(S)[1] / (1 + e) / 1e4:.1f} 万 token/s")

## 3. 代码里才看得到的

1. `t_grad` 对每种超节点在分层和连续环之间取较快者：出口固定 400 GB/s 的 64 卡里，连续环（317.7 ms）比分层（331.1 ms）快。
2. 128、256 卡的吞吐代码算出 156.1、158.2 万，原文 156.0、158.1：末位舍入不同。
3. 恢复时间里的 $S \times 64$ GB ÷ 64 GB/s：受影响超节点每卡 64 GB 状态经一条 PCIe Gen5 读回，所以恰好是 $S$ 秒。

## 4. 自己改一改

- 把本地带宽改成 900 GB/s（第五代 NVLink），128 卡步时间变多少？
- 给 `step_1024` 加 3:1 超售参数。
- 把 checkpoint 间隔从 600 s 扫到 3600 s，找 128 卡时开销最小的间隔。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 四人约出发看最晚那个，车开快不如让他准时；
> 装盘等菜和汤，汤早好了再快也没用；
> 搬家换大车，寄信少跑邮局；
> 部门大了跨部门会少，大到头就只剩出错时牵连的人多。